# Feature Engineering - Extract Features to CSV

This notebook extracts comprehensive features from the transaction graph and exports them to CSV format for machine learning.

## Feature Categories:
1. **Frequency Features (6)**: Transaction frequency over time windows
2. **Statistical Features (21)**: Account behavior, temporal patterns, amounts
3. **Centrality Features (7)**: Graph-based network metrics

## Input:
- `../dataset/MulDiGraph/output_transactions.txt` - Flat transaction records

## Output:
- `../dataset/MulDiGraph/features.csv` - Feature matrix with 34 features per node

## Step 1: Import Required Libraries

In [3]:
import sys
sys.path.append('../features_engineering')

import pandas as pd
import numpy as np
import directed_freq_loader
import directed_stat_loader
import networkx as nx
import scipy.sparse as sp
from tqdm import tqdm

## Step 2: Configure Parameters

Set the dataset name and file paths for input and output.

In [4]:
# Configuration
DATASET = "MulDiGraph"
INPUT_PATH = f"../data/dataset/MulDiGraph/output_transactions.txt"
OUTPUT_PATH = f"../data/dataset/Data_after_FE/features.csv"
ALPHA = 1.0  # Temporal decay factor for centrality calculation

print(f"Dataset: {DATASET}")
print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Alpha (temporal decay): {ALPHA}")

Dataset: MulDiGraph
Input: ../data/dataset/MulDiGraph/output_transactions.txt
Output: ../data/dataset/Data_after_FE/features.csv
Alpha (temporal decay): 1.0


## Step 3: Load Transaction Data

In [5]:
# Read transaction data
print("Reading input data...")
data = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(data)} transactions")
print(f"Columns: {list(data.columns)}")
print("\nFirst 5 rows:")
print(data.head())

Reading input data...
Loaded 3345322 transactions
Columns: ['303882', '571920', '1508251757', '0.0']

First 5 rows:
   303882  571920  1508251757  0.0
0  303882  571920  1508251765  0.0
1  303882  571920  1508251765  0.0
2  303882  571920  1508251765  0.0
3  303882  482988  1508253708  0.0
4  303882  482988  1508303187  0.0


## Step 4: Initialize Graph Loaders

Create two loader instances to build temporal directed graphs.

In [6]:
# Initialize frequency loader
print("Initializing frequency loader...")
freq_loader = directed_freq_loader.directed_loader()
freq_loader.read(data)
nv = len(freq_loader.G)
print(f"Frequency loader initialized with {nv} nodes")

# Initialize statistical loader
print("\nInitializing statistical loader...")
stat_loader = directed_stat_loader.directed_loader()
stat_loader.read(data)
print(f"Statistical loader initialized with {len(stat_loader.G)} nodes")

INFO:directed_freq_loader:Processing 3345322 transactions...


Initializing frequency loader...


INFO:directed_freq_loader:Finalizing graph structure...
INFO:directed_freq_loader:Graph built with 825820 unique nodes
INFO:directed_stat_loader:Processing 3345322 transactions with amounts...


Frequency loader initialized with 825820 nodes

Initializing statistical loader...


INFO:directed_stat_loader:Finalizing graph structure...
INFO:directed_stat_loader:Graph built with 825820 unique nodes


Statistical loader initialized with 825820 nodes


## Step 5: Calculate Frequency Features

Compute time-windowed transfer frequency features (long-term and short-term).

### 📊 Understanding Frequency Features (6 Features)

Frequency features capture **temporal transaction patterns** over different time windows. These features help identify bot-like behavior and burst attack patterns commonly seen in phishing accounts.

**Features calculated:**
- **Long-term transfer frequency** (30 days): Total transaction rate over extended period
- **Short-term transfer frequency** (7 days): Recent activity burst detection
- **Long-term incoming/outgoing frequency**: Separate tracking of inbound vs outbound patterns
- **Short-term incoming/outgoing frequency**: Recent directional transaction spikes

**Why we need this:** Phishing accounts often exhibit unusual burst patterns (high short-term frequency) followed by inactivity, while normal accounts maintain more consistent transaction rates. The distinction between incoming and outgoing frequencies helps detect "funnel flow" attacks where phishers receive many small transactions but quickly consolidate funds.

In [7]:
print("Calculating frequency features...")
freq_features = freq_loader.cal_stat_feats()
print(f"Frequency features calculated for {len(freq_features)} nodes")

# Display sample features
print("\nSample frequency features (node 0):")
if 0 in freq_features:
    for key, value in freq_features[0].items():
        print(f"  {key}: {value}")

Calculating frequency features...
Frequency features calculated for 825820 nodes

Sample frequency features (node 0):
  Long-term transfer frequency: 0.0
  Short-term transfer frequency: 0.0
  Long-term incoming transfer frequency: 0.0
  Short-term incoming transfer frequency: 0.0
  Long-term outgoing transfer frequency: 0.0
  Short-term outgoing transfer frequency: 0.0


## Step 6: Calculate Statistical Features

Compute behavioral and temporal statistical features per node.

### 📈 Understanding Statistical Features (21 Features)

Statistical features capture **account behavior patterns** including transaction amounts, temporal patterns, and account characteristics. These features distinguish normal trading behavior from fraudulent activity.

**Feature categories:**

**1. Node Degree Features (3):**
- **node_indegree / node_outdegree**: Number of unique incoming/outgoing connections
- **direction_ratio**: Ratio of incoming to outgoing transactions
- *Purpose:* Phishers often have high indegree (many victims) with low outdegree (few consolidation addresses)

**2. Transaction Amount Features (7):**
- **max/min/average incoming/outgoing amounts**: Transaction value statistics
- **account_balance**: Current balance (incoming - outgoing)
- *Purpose:* Phishers typically receive many small amounts and quickly move large consolidated sums

**3. Temporal Pattern Features (7):**
- **account_lifetime**: Days since first transaction
- **active_days**: Number of days with at least one transaction
- **mean_hour_sent/received**: Average hour of day for transactions (0-23)
- **std_hour_sent/received**: Standard deviation showing consistency in timing
- *Purpose:* Automated phishing bots show consistent timing patterns; legitimate users vary more

**4. Transaction Timing Features (3):**
- **avg/min/max_time_between_tx**: Time gaps between consecutive transactions
- *Purpose:* Bot behavior shows regular intervals; human behavior is more irregular

**5. Weekend Pattern Features (2):**
- **wd_tx_ratio_sent/received**: Ratio of weekday to weekend transactions
- *Purpose:* Automated attacks continue on weekends; human activity typically decreases

**Why we need this:** These 21 features capture the multi-dimensional behavioral signature of accounts. Phishing accounts exhibit systematic patterns (regular timing, high incoming ratio, quick fund movement) that differ markedly from normal trading behavior.

In [8]:
print("Calculating statistical features...")
stat_features = stat_loader.cal_stat_feats()
print(f"Statistical features calculated for {len(stat_features)} nodes")

# Display sample features
print("\nSample statistical features (node 0):")
if 0 in stat_features:
    for key, value in list(stat_features[0].items())[:10]:  # Show first 10
        print(f"  {key}: {value}")

INFO:directed_stat_loader:Calculating statistical features for 825820 nodes...


Calculating statistical features...
Statistical features calculated for 825820 nodes

Sample statistical features (node 0):
  node_outdegree: 11
  node_indegree: 7
  direction_ratio: 0.636363578512402
  total_val_sent: 0.0
  max_outgoing_amount: 0.0
  min_outgoing_amount: 0.0
  max_incoming_amount: 49.9995
  min_incoming_amount: 4.025144
  average_outgoing_amount: 0.0
  average_incoming_amount: 22.22679377777778


## Step 7: Define Centrality Computation Functions

In [9]:
def construct_directed_adjacency_matrix(G, nv, alpha):
    """Construct directed adjacency matrix preserving edge directions."""
    row, col, data = [], [], []
    
    for v in range(nv):
        for (t, lii, lio) in G[v]:
            weight = np.exp(-t / alpha)
            
            lii_list = lii.tolist() if lii is not None and lii.size > 0 else []
            for u in lii_list:
                row.append(u)
                col.append(v)
                data.append(weight)
            
            lio_list = lio.tolist() if lio is not None and lio.size > 0 else []
            for u in lio_list:
                row.append(v)
                col.append(u)
                data.append(weight)
    
    A_directed = sp.csr_matrix((data, (row, col)), shape=(nv, nv))
    A_directed = (A_directed > 0).astype(np.float32)
    
    return A_directed

def compute_centrality_features(G, nv, alpha):
    """Compute centrality features efficiently."""
    A_directed = construct_directed_adjacency_matrix(G, nv, alpha)
    A_nx_directed = nx.from_scipy_sparse_array(A_directed, parallel_edges=False, 
                                               edge_attribute="weight", create_using=nx.DiGraph)
    
    features = {}
    
    print(f"Computing centrality features for {nv} nodes...")
    
    print("Computing Katz centrality...")
    try:
        features['katz'] = nx.katz_centrality(A_nx_directed, alpha=0.01, beta=1.0, max_iter=1000, tol=1e-6)
    except:
        features['katz'] = {i: 0.0 for i in range(nv)}

    print("Computing Degree centrality...")
    features['degree'] = nx.degree_centrality(A_nx_directed)

    print("Computing Closeness centrality...")
    features['closeness'] = nx.closeness_centrality(A_nx_directed)

    print("Computing Clustering coefficient...")
    features['clustering'] = nx.clustering(A_nx_directed)

    print("Computing Eigenvector centrality...")
    try:
        features['eigenvector'] = nx.eigenvector_centrality(A_nx_directed, max_iter=1000, tol=1e-4)
    except:
        features['eigenvector'] = nx.pagerank(A_nx_directed, max_iter=1000, tol=1e-4)

    print("Computing In-degree centrality...")
    features['indegree'] = nx.in_degree_centrality(A_nx_directed)
    
    print("Computing Out-degree centrality...")
    features['outdegree'] = nx.out_degree_centrality(A_nx_directed)
    
    print("Centrality computation completed.")
    return features

print("Centrality computation functions defined.")

Centrality computation functions defined.


## Step 8: Compute Graph Centrality Features

Calculate network centrality metrics using NetworkX.

### 🌐 Understanding Centrality Features (7 Features)

Centrality features measure an account's **position and importance within the transaction network**. These graph-based metrics reveal structural patterns that distinguish isolated phishing operations from legitimate trading activity.

**Features calculated:**

**1. Katz Centrality:**
- Measures influence by counting all paths to a node with exponential decay
- *Purpose:* Phishing accounts often have low Katz centrality (victims don't connect onward)

**2. Degree Centrality:**
- Normalized count of direct connections
- *Purpose:* Identifies hub nodes in the network

**3. Closeness Centrality:**
- Measures how close a node is to all other nodes (average shortest path)
- *Purpose:* Phishing accounts tend to be peripherally located, far from network center

**4. Clustering Coefficient:**
- Measures how well a node's neighbors are connected to each other
- *Purpose:* Phishing victims typically don't interact with each other (low clustering)

**5. Eigenvector Centrality:**
- Measures importance based on having connections to other important nodes
- *Purpose:* Legitimate traders connect to established accounts; phishers connect to isolated victims

**6. In-degree Centrality:**
- Normalized incoming connections
- *Purpose:* High in-degree with low out-degree suggests funnel pattern

**7. Out-degree Centrality:**
- Normalized outgoing connections
- *Purpose:* Low out-degree indicates rapid fund consolidation

**Why we need this:** Network topology reveals attack structures. Phishing accounts create characteristic "funnel" patterns (many victims → phisher → few consolidation addresses) that are invisible in transaction-level features alone. These 7 centrality metrics capture the graph structure that distinguishes coordinated attacks from organic trading networks.

In [10]:
print("Computing graph centrality features...")
graph_features = compute_centrality_features(freq_loader.G, nv, ALPHA)
print("\nCentrality features computed successfully!")

Computing graph centrality features...
Computing centrality features for 825820 nodes...
Computing Katz centrality...
Computing Degree centrality...
Computing Closeness centrality...
Computing Clustering coefficient...
Computing Eigenvector centrality...
Computing In-degree centrality...
Computing Out-degree centrality...
Centrality computation completed.

Centrality features computed successfully!


## Step 9: Merge All Features into DataFrame

Combine frequency, statistical, and centrality features for each node.

### 🔗 Feature Integration: Combining 34 Features

Now we combine all three feature categories into a unified feature matrix for machine learning:

**Feature composition:**
- **6 Frequency features** → Temporal patterns and transaction bursts
- **21 Statistical features** → Behavioral characteristics and account properties  
- **7 Centrality features** → Network position and graph structure

**Total: 34 features per node**

This multi-dimensional approach ensures we capture phishing behavior from multiple angles:
- ⏰ **When** transactions occur (frequency)
- 💰 **How** transactions behave (statistical)
- 🕸️ **Where** accounts sit in the network (centrality)

Each feature category complements the others - a phisher might appear normal in one dimension but exhibit suspicious patterns in another.

In [11]:
print("Preparing data for CSV export...")
rows = []

for v in tqdm(range(nv), desc="Processing nodes"):
    row = {'node_id': int(freq_loader.revco[v])}
    
    # Add frequency features (6 dimensions)
    if v in freq_features:
        row['long_term_transfer_freq'] = freq_features[v]['Long-term transfer frequency']
        row['short_term_transfer_freq'] = freq_features[v]['Short-term transfer frequency']
        row['long_term_incoming_freq'] = freq_features[v]['Long-term incoming transfer frequency']
        row['short_term_incoming_freq'] = freq_features[v]['Short-term incoming transfer frequency']
        row['long_term_outgoing_freq'] = freq_features[v]['Long-term outgoing transfer frequency']
        row['short_term_outgoing_freq'] = freq_features[v]['Short-term outgoing transfer frequency']
    else:
        row['long_term_transfer_freq'] = 0.0
        row['short_term_transfer_freq'] = 0.0
        row['long_term_incoming_freq'] = 0.0
        row['short_term_incoming_freq'] = 0.0
        row['long_term_outgoing_freq'] = 0.0
        row['short_term_outgoing_freq'] = 0.0
    
    # Add statistical features (21 dimensions)
    if v in stat_features:
        row['node_indegree'] = stat_features[v]['node_indegree']
        row['direction_ratio'] = stat_features[v]['direction_ratio']
        row['max_outgoing_amount'] = stat_features[v]['max_outgoing_amount']
        row['min_outgoing_amount'] = stat_features[v]['min_outgoing_amount']
        row['average_outgoing_amount'] = stat_features[v]['average_outgoing_amount']
        row['account_balance'] = stat_features[v]['account_balance']
        row['mean_hour_received'] = stat_features[v]['mean_hour_received']
        row['std_hour_received'] = stat_features[v]['std_hour_received']
        row['min_time_between_tx'] = stat_features[v]['min_time_between_tx']
        row['wd_tx_ratio_received'] = stat_features[v]['wd_tx_ratio_received']
        
        # Additional statistical features
        row['node_outdegree'] = stat_features[v]['node_outdegree']
        row['max_incoming_amount'] = stat_features[v]['max_incoming_amount']
        row['min_incoming_amount'] = stat_features[v]['min_incoming_amount']
        row['average_incoming_amount'] = stat_features[v]['average_incoming_amount']
        row['account_lifetime'] = stat_features[v]['account_lifetime']
        row['active_days'] = stat_features[v]['active_days']
        row['mean_hour_sent'] = stat_features[v]['mean_hour_sent']
        row['std_hour_sent'] = stat_features[v]['std_hour_sent']
        row['avg_time_between_tx'] = stat_features[v]['avg_time_between_tx']
        row['max_time_between_tx'] = stat_features[v]['max_time_between_tx']
        row['wd_tx_ratio_sent'] = stat_features[v]['wd_tx_ratio_sent']
    else:
        # Default values if node not found in stat_features
        stat_cols = ['node_indegree', 'direction_ratio', 'max_outgoing_amount', 'min_outgoing_amount',
                    'average_outgoing_amount', 'account_balance', 'mean_hour_received', 'std_hour_received',
                    'min_time_between_tx', 'wd_tx_ratio_received', 'node_outdegree', 'max_incoming_amount',
                    'min_incoming_amount', 'average_incoming_amount', 'account_lifetime', 'active_days',
                    'mean_hour_sent', 'std_hour_sent', 'avg_time_between_tx', 'max_time_between_tx', 'wd_tx_ratio_sent']
        for col in stat_cols:
            row[col] = 0.0
    
    # Add centrality features (7 dimensions)
    row['katz_centrality'] = graph_features['katz'].get(v, 0.0)
    row['degree_centrality'] = graph_features['degree'].get(v, 0.0)
    row['closeness_centrality'] = graph_features['closeness'].get(v, 0.0)
    row['clustering_coefficient'] = graph_features['clustering'].get(v, 0.0)
    row['eigenvector_centrality'] = graph_features['eigenvector'].get(v, 0.0)
    row['indegree_centrality'] = graph_features['indegree'].get(v, 0.0)
    row['outdegree_centrality'] = graph_features['outdegree'].get(v, 0.0)
    
    rows.append(row)

print(f"\nProcessed {len(rows)} nodes")

Preparing data for CSV export...


Processing nodes: 100%|██████████| 825820/825820 [00:07<00:00, 108966.03it/s]


Processed 825820 nodes


## Step 10: Create DataFrame and Reorder Columns

In [12]:
print("Creating DataFrame...")
df = pd.DataFrame(rows)

# Reorder columns for better readability
freq_cols = ['long_term_transfer_freq', 'short_term_transfer_freq', 'long_term_incoming_freq', 
            'short_term_incoming_freq', 'long_term_outgoing_freq', 'short_term_outgoing_freq']

selected_stat_cols = ['node_indegree', 'direction_ratio', 'max_outgoing_amount', 'min_outgoing_amount',
                     'average_outgoing_amount', 'account_balance', 'mean_hour_received', 'std_hour_received',
                     'min_time_between_tx', 'wd_tx_ratio_received']

other_stat_cols = ['node_outdegree', 'max_incoming_amount', 'min_incoming_amount', 'average_incoming_amount',
                  'account_lifetime', 'active_days', 'mean_hour_sent', 'std_hour_sent', 'avg_time_between_tx', 
                  'max_time_between_tx', 'wd_tx_ratio_sent']

centrality_cols = ['katz_centrality', 'degree_centrality', 'closeness_centrality', 'clustering_coefficient',
                  'eigenvector_centrality', 'indegree_centrality', 'outdegree_centrality']

column_order = ['node_id'] + freq_cols + selected_stat_cols + other_stat_cols + centrality_cols
df = df[column_order]

print(f"DataFrame shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

Creating DataFrame...
DataFrame shape: (825820, 35)
Columns: 35


## Step 11: Save to CSV and Display Summary

In [13]:
# Save to CSV
df.to_csv(OUTPUT_PATH, index=False)

print("="*60)
print("FEATURE EXTRACTION COMPLETED")
print("="*60)
print(f"Features exported to: {OUTPUT_PATH}")
print(f"Dataset shape: {df.shape}")
print("\nColumn summary:")
print(f"- Node ID: 1 column")
print(f"- Frequency features: {len(freq_cols)} columns")
print(f"- Selected statistical features: {len(selected_stat_cols)} columns")
print(f"- Other statistical features: {len(other_stat_cols)} columns")
print(f"- Centrality features: {len(centrality_cols)} columns")
print(f"- Total features: {len(df.columns)-1} columns")
print("="*60)

FEATURE EXTRACTION COMPLETED
Features exported to: ../data/dataset/Data_after_FE/features.csv
Dataset shape: (825820, 35)

Column summary:
- Node ID: 1 column
- Frequency features: 6 columns
- Selected statistical features: 10 columns
- Other statistical features: 11 columns
- Centrality features: 7 columns
- Total features: 34 columns


## Step 12: Display Sample Data and Statistics

In [14]:
# Show first few rows
print("First 5 rows:")
print(df.head())

print("\n" + "="*60)
print("Feature statistics:")
print(df.describe())

First 5 rows:
   node_id  long_term_transfer_freq  short_term_transfer_freq  \
0   303882                 0.000000                  0.000000   
1   571920                 0.000000                  0.000000   
2   482988                 0.000000                  0.000000   
3   122138                 0.033333                  0.000000   
4   443788                 0.266667                  0.285714   

   long_term_incoming_freq  short_term_incoming_freq  long_term_outgoing_freq  \
0                 0.000000                  0.000000                 0.000000   
1                 0.000000                  0.000000                 0.000000   
2                 0.000000                  0.000000                 0.000000   
3                 0.000000                  0.000000                 0.033333   
4                 0.266667                  0.285714                 0.000000   

   short_term_outgoing_freq  node_indegree  direction_ratio  \
0                       0.0              7   